# ⛏️ Minecraft Case Opener — Kaggle Edition

A complete, self-contained Minecraft-themed case-opening simulator built with **Python + Gradio**.

Run every cell top-to-bottom (Kaggle: *Run All*). The last cell launches a Gradio web app
with a shareable link.

**Features:** Home, Case Shop, Case Opening (CS:GO-style scroll animation), Inventory,
Statistics, Settings — all backed by a JSON save file, with a virtual-coin economy
starting at **1,000,000 coins**.

## 1. Install & Import Dependencies
Installs `gradio` (and pins a compatible version) if not already present.

In [ ]:
# --------------------------------------------------------------------------
# 1. INSTALL DEPENDENCIES
# --------------------------------------------------------------------------
import sys, subprocess

def _install(pkg):
    """Install a pip package quietly, ignoring failures from already-satisfied reqs."""
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
    except subprocess.CalledProcessError as e:
        print(f"Warning: could not install {pkg}: {e}")

try:
    import gradio  # noqa
    print(f"gradio already installed (version {gradio.__version__})")
except ImportError:
    print("Installing gradio ...")
    _install("gradio>=4.36.0")

print("Dependency check complete.")

In [ ]:
# --------------------------------------------------------------------------
# 2. IMPORTS
# --------------------------------------------------------------------------
import json
import os
import random
import time
import uuid
import copy
from dataclasses import dataclass, field, asdict
from datetime import datetime
from typing import List, Dict, Optional, Tuple

import gradio as gr

print("Imports successful. Gradio version:", gr.__version__)

## 2. Data Model — Items, Rarities, and Cases
Defines every Minecraft-themed item, its value, rarity tier, and drop weight, plus the six cases and their drop tables.

In [ ]:
# --------------------------------------------------------------------------
# 3. RARITY TIERS
# --------------------------------------------------------------------------
# Each rarity has a display name, a color (for the UI), and a relative "weight
# class" used only for sorting/statistics (actual drop odds come from each
# case's own drop table, defined further below).

RARITY_ORDER = [
    "Common", "Uncommon", "Rare", "Epic", "Legendary", "Mythic", "Divine"
]

RARITY_COLORS = {
    "Common":    "#b0b0b0",
    "Uncommon":  "#55ff55",
    "Rare":      "#55ffff",
    "Epic":      "#aa55ff",
    "Legendary": "#ffaa00",
    "Mythic":    "#ff5555",
    "Divine":    "#ffe680",
}

def rarity_rank(rarity: str) -> int:
    """Return numeric rank of a rarity (higher = rarer) for sorting."""
    try:
        return RARITY_ORDER.index(rarity)
    except ValueError:
        return -1


# --------------------------------------------------------------------------
# 4. MASTER ITEM POOL
# --------------------------------------------------------------------------
# value is in raw coins. k = 1,000 | m = 1,000,000 | b = 1,000,000,000
# Rarity tiers are assigned based on value (higher value -> rarer tier).

def _v(n):
    """Helper to keep the table readable; n is already the raw coin value."""
    return int(n)

MASTER_ITEMS = [
    # name,                    value,              rarity,      emoji
    ("Command Block",          _v(10_000_000_000),  "Divine",    "🟪"),
    ("Dragon Egg",             _v(5_000_000_000),   "Divine",    "🥚"),
    ("Ender Dragon",           _v(2_000_000_000),   "Mythic",    "🐉"),
    ("Villager Dollar Bill",   _v(1_000_000_000),   "Mythic",    "💵"),
    ("Arrow of Luck",          _v(500_000_000),     "Mythic",    "🏹"),
    ("End Gateway",            _v(400_000_000),     "Legendary", "🌀"),
    ("Elytra",                 _v(300_000_000),     "Legendary", "🦋"),
    ("Barrier Block",          _v(200_000_000),     "Legendary", "🚧"),
    ("Bedrock",                _v(100_000_000),     "Legendary", "⬛"),
    ("Emerald5",               _v(100_000_000),     "Legendary", "💎"),
    ("Netherite Block",        _v(50_000_000),      "Epic",      "🟫"),
    ("Emerald4",               _v(50_000_000),      "Epic",      "💚"),
    ("Amethyst Key",           _v(40_000_000),       "Epic",      "🔑"),
    ("Dragon Head",            _v(30_000_000),       "Epic",      "🐲"),
    ("Wither",                 _v(25_000_000),       "Epic",      "☠️"),
    ("Emerald3",               _v(10_000_000),       "Rare",      "💚"),
    ("Debug Stick",            _v(10_000_000),       "Rare",      "🪄"),
    ("Trident",                _v(6_000_000),        "Rare",      "🔱"),
    ("Skeleton Spawner",       _v(5_000_000),        "Rare",      "💀"),
    ("Villager",               _v(3_000_000),        "Uncommon",  "🧑"),
    ("Netherite Ingot",        _v(2_500_000),        "Uncommon",  "🔶"),
    ("Ancient Debris",         _v(2_000_000),        "Uncommon",  "🪨"),
    ("Gilded Blackstone",      _v(1_000_000),        "Uncommon",  "⬛"),
    ("Emerald2",               _v(1_000_000),        "Uncommon",  "💚"),
    ("Enchanted Apple",        _v(650_000),          "Uncommon",  "🍎"),
    ("Trial Spawner",          _v(500_000),          "Common",    "🔲"),
    ("Emerald1",               _v(100_000),          "Common",    "💚"),
    ("Diamond",                _v(50_000),           "Common",    "💠"),
    ("Shulker",                _v(50_000),           "Common",    "📦"),
    ("Coal",                   _v(20_000),           "Common",    "⚫"),
    ("Dirt",                   _v(1_000),            "Common",    "🟤"),
    ("Netherrack",             _v(1_000),            "Common",    "🟥"),
]

# Build a lookup dict: name -> item info
ITEM_DB: Dict[str, Dict] = {
    name: {"name": name, "value": value, "rarity": rarity, "emoji": emoji}
    for (name, value, rarity, emoji) in MASTER_ITEMS
}

print(f"Loaded {len(ITEM_DB)} unique items across {len(RARITY_ORDER)} rarity tiers.")

In [ ]:
# --------------------------------------------------------------------------
# 5. CASE DEFINITIONS
# --------------------------------------------------------------------------
# Each case has: display name, price, emoji/icon, and a drop table of
# (item_name, weight). Weights are relative probabilities *within that case*.
# Cheaper cases skew toward common/uncommon items with tiny chances at
# something big; expensive cases skew toward epic/legendary/mythic/divine.

def _table(*pairs):
    """Build a drop table dict from (name, weight) pairs."""
    return {name: weight for name, weight in pairs}

CASES: Dict[str, Dict] = {
    "Starter Case": {
        "price": 5_000,
        "icon": "📦",
        "description": "A humble beginning. Mostly common junk, but every miner starts somewhere.",
        "drop_table": _table(
            ("Dirt", 350),
            ("Netherrack", 300),
            ("Coal", 180),
            ("Shulker", 60),
            ("Diamond", 55),
            ("Emerald1", 40),
            ("Trial Spawner", 12),
            ("Enchanted Apple", 3),
        ),
    },
    "Miner Case": {
        "price": 25_000,
        "icon": "⛏️",
        "description": "For the dedicated digger. Better odds at emeralds and ancient debris.",
        "drop_table": _table(
            ("Coal", 260),
            ("Shulker", 180),
            ("Diamond", 170),
            ("Emerald1", 140),
            ("Trial Spawner", 100),
            ("Enchanted Apple", 70),
            ("Emerald2", 40),
            ("Gilded Blackstone", 22),
            ("Ancient Debris", 12),
            ("Netherite Ingot", 5),
            ("Villager", 1),
        ),
    },
    "Nether Case": {
        "price": 100_000,
        "icon": "🔥",
        "description": "Crack this open in the Nether. Netherite and Wither-tier loot await.",
        "drop_table": _table(
            ("Emerald1", 220),
            ("Diamond", 180),
            ("Enchanted Apple", 150),
            ("Emerald2", 130),
            ("Gilded Blackstone", 100),
            ("Ancient Debris", 80),
            ("Netherite Ingot", 60),
            ("Villager", 40),
            ("Skeleton Spawner", 20),
            ("Trident", 10),
            ("Debug Stick", 6),
            ("Emerald3", 3),
            ("Wither", 1),
        ),
    },
    "End Case": {
        "price": 500_000,
        "icon": "🌌",
        "description": "Beyond the void portal. High-tier loot with a shot at End Gateways.",
        "drop_table": _table(
            ("Ancient Debris", 150),
            ("Netherite Ingot", 130),
            ("Villager", 110),
            ("Skeleton Spawner", 90),
            ("Trident", 70),
            ("Debug Stick", 55),
            ("Emerald3", 45),
            ("Wither", 25),
            ("Dragon Head", 12),
            ("Amethyst Key", 8),
            ("Emerald4", 4),
            ("Netherite Block", 3),
            ("Bedrock", 1.5),
            ("Emerald5", 1),
            ("Barrier Block", 0.5),
        ),
    },
    "Legendary Case": {
        "price": 2_000_000,
        "icon": "🏆",
        "description": "Only the bold spend here. Loaded odds toward Legendary-tier gear.",
        "drop_table": _table(
            ("Skeleton Spawner", 110),
            ("Trident", 100),
            ("Debug Stick", 90),
            ("Emerald3", 80),
            ("Wither", 60),
            ("Dragon Head", 45),
            ("Amethyst Key", 35),
            ("Emerald4", 28),
            ("Netherite Block", 22),
            ("Bedrock", 12),
            ("Emerald5", 10),
            ("Barrier Block", 6),
            ("Elytra", 3),
            ("End Gateway", 1.5),
            ("Arrow of Luck", 0.4),
            ("Villager Dollar Bill", 0.1),
        ),
    },
    "Mythic Case": {
        "price": 10_000_000,
        "icon": "🐲",
        "description": "The ultimate gamble. Dragon Eggs and Command Blocks lie in wait.",
        "drop_table": _table(
            ("Amethyst Key", 90),
            ("Dragon Head", 80),
            ("Wither", 70),
            ("Emerald4", 60),
            ("Netherite Block", 50),
            ("Bedrock", 40),
            ("Emerald5", 35),
            ("Barrier Block", 28),
            ("Elytra", 20),
            ("End Gateway", 14),
            ("Arrow of Luck", 8),
            ("Villager Dollar Bill", 3),
            ("Ender Dragon", 1.2),
            ("Dragon Egg", 0.4),
            ("Command Block", 0.15),
        ),
    },
}

# Sanity check: every item referenced in a drop table must exist in ITEM_DB
for case_name, case_info in CASES.items():
    for item_name in case_info["drop_table"]:
        assert item_name in ITEM_DB, f"Unknown item '{item_name}' referenced in {case_name}"

print(f"Loaded {len(CASES)} cases.")

## 3. Save/Load System
Handles the player's persistent JSON save file (auto-created if missing) with graceful error handling.

In [ ]:
# --------------------------------------------------------------------------
# 6. SAVE FILE MANAGEMENT
# --------------------------------------------------------------------------
SAVE_PATH = "/kaggle/working/mc_case_save.json"
# Fallback for non-Kaggle environments (e.g. local testing)
if not os.path.isdir("/kaggle/working"):
    SAVE_PATH = os.path.join(os.getcwd(), "mc_case_save.json")

STARTING_BALANCE = 1_000_000

def default_save_data() -> dict:
    """Return a fresh save-file structure for a brand-new player."""
    return {
        "balance": STARTING_BALANCE,
        "inventory": [],          # list of item dicts (each with a unique instance id)
        "cases_opened": 0,
        "coins_spent": 0,
        "coins_earned": 0,
        "best_item": None,        # dict of the highest-value item ever obtained
        "history": [],            # list of recent open events (capped)
        "settings": {
            "animation_speed": "Normal",   # Slow / Normal / Fast
            "sound_enabled": True,
            "confirm_sales": True,
        },
        "created_at": datetime.utcnow().isoformat(),
    }


def load_save() -> dict:
    """
    Load the save file from disk, creating it with defaults if it does not
    exist or is corrupted. Never raises — always returns a usable dict.
    """
    if not os.path.exists(SAVE_PATH):
        data = default_save_data()
        write_save(data)
        return data

    try:
        with open(SAVE_PATH, "r", encoding="utf-8") as f:
            data = json.load(f)
        # Backfill any missing keys (e.g. if the save schema grows over time)
        defaults = default_save_data()
        for key, val in defaults.items():
            if key not in data:
                data[key] = val
        for key, val in defaults["settings"].items():
            if key not in data.get("settings", {}):
                data.setdefault("settings", {})[key] = val
        return data
    except (json.JSONDecodeError, OSError, ValueError) as e:
        print(f"Save file corrupted or unreadable ({e}); creating a fresh save.")
        data = default_save_data()
        write_save(data)
        return data


def write_save(data: dict) -> None:
    """Persist the given save dict to disk, handling I/O errors gracefully."""
    try:
        tmp_path = SAVE_PATH + ".tmp"
        with open(tmp_path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)
        os.replace(tmp_path, SAVE_PATH)
    except OSError as e:
        print(f"Warning: failed to write save file: {e}")


# Load (or create) the save at notebook start
STATE = load_save()
print(f"Save file location: {SAVE_PATH}")
print(f"Loaded player state — balance: {STATE['balance']:,} coins, "
      f"inventory size: {len(STATE['inventory'])}, cases opened: {STATE['cases_opened']}")

## 4. Core Game Logic
Classes and functions for opening cases, managing inventory, and computing statistics. All UI callbacks build on this layer.

In [ ]:
# --------------------------------------------------------------------------
# 7. FORMATTING HELPERS
# --------------------------------------------------------------------------
def format_coins(n) -> str:
    """Format a raw coin value into a compact, readable string, e.g. 1_500_000 -> '1.5M'."""
    n = float(n)
    sign = "-" if n < 0 else ""
    n = abs(n)
    if n >= 1_000_000_000:
        return f"{sign}{n/1_000_000_000:.2f}B"
    if n >= 1_000_000:
        return f"{sign}{n/1_000_000:.2f}M"
    if n >= 1_000:
        return f"{sign}{n/1_000:.2f}K"
    return f"{sign}{n:,.0f}"


def format_coins_full(n) -> str:
    """Format a raw coin value with full comma separation, e.g. 1500000 -> '1,500,000'."""
    return f"{int(n):,}"


def rarity_badge(rarity: str) -> str:
    """Return a small HTML badge for a rarity tier."""
    color = RARITY_COLORS.get(rarity, "#ffffff")
    return f'<span style="color:{color}; font-weight:700;">{rarity}</span>'


# --------------------------------------------------------------------------
# 8. GAME ENGINE CLASS
# --------------------------------------------------------------------------
class GameEngine:
    """
    Encapsulates all game logic: economy, case opening, inventory management,
    and statistics. Operates on the module-level STATE dict and persists
    changes to disk after every mutating action.
    """

    def __init__(self, state: dict):
        self.state = state

    # ---------------------------------------------------------------- utils
    def save(self):
        write_save(self.state)

    @property
    def balance(self) -> int:
        return int(self.state["balance"])

    def inventory_value(self) -> int:
        return sum(item["value"] for item in self.state["inventory"])

    def profit_loss(self) -> int:
        return int(self.state["coins_earned"] - self.state["coins_spent"])

    # ------------------------------------------------------------ case shop
    def can_afford(self, case_name: str) -> bool:
        case = CASES.get(case_name)
        return bool(case) and self.balance >= case["price"]

    def roll_item(self, case_name: str) -> dict:
        """Weighted-random pick of one item name from a case's drop table."""
        case = CASES[case_name]
        table = case["drop_table"]
        names = list(table.keys())
        weights = list(table.values())
        chosen_name = random.choices(names, weights=weights, k=1)[0]
        base = ITEM_DB[chosen_name]
        return copy.deepcopy(base)

    def open_case(self, case_name: str) -> Tuple[bool, str, Optional[dict]]:
        """
        Attempt to open a case: deduct price, roll an item, add to inventory,
        and update statistics. Returns (success, message, item_or_None).
        """
        if case_name not in CASES:
            return False, f"Unknown case: {case_name}", None

        case = CASES[case_name]
        price = case["price"]

        if self.balance < price:
            return False, "Insufficient balance to open this case.", None

        # Deduct price
        self.state["balance"] -= price
        self.state["coins_spent"] += price

        # Roll item
        item = self.roll_item(case_name)
        item["id"] = str(uuid.uuid4())
        item["obtained_at"] = datetime.utcnow().isoformat()
        item["from_case"] = case_name

        # Add to inventory
        self.state["inventory"].append(item)
        self.state["cases_opened"] += 1

        # Track best item ever obtained
        best = self.state.get("best_item")
        if best is None or item["value"] > best["value"]:
            self.state["best_item"] = {
                "name": item["name"],
                "value": item["value"],
                "rarity": item["rarity"],
                "emoji": item["emoji"],
            }

        # Track recent history (cap at 50 entries)
        self.state["history"].insert(0, {
            "case": case_name,
            "item": item["name"],
            "rarity": item["rarity"],
            "value": item["value"],
            "time": item["obtained_at"],
        })
        self.state["history"] = self.state["history"][:50]

        self.save()
        return True, f"Obtained {item['name']}!", item

    # ------------------------------------------------------------ inventory
    def find_item(self, item_id: str) -> Optional[dict]:
        for item in self.state["inventory"]:
            if item["id"] == item_id:
                return item
        return None

    def sell_item(self, item_id: str) -> Tuple[bool, str, int]:
        """Sell a single inventory item by its unique id. Returns (ok, msg, value)."""
        item = self.find_item(item_id)
        if item is None:
            return False, "Item not found in inventory.", 0

        value = int(item["value"])
        self.state["inventory"] = [i for i in self.state["inventory"] if i["id"] != item_id]
        self.state["balance"] += value
        self.state["coins_earned"] += value
        self.save()
        return True, f"Sold {item['name']} for {format_coins_full(value)} coins.", value

    def sell_all(self) -> Tuple[bool, str, int]:
        """Sell every item currently in the inventory."""
        if not self.state["inventory"]:
            return False, "Inventory is already empty.", 0

        total = sum(i["value"] for i in self.state["inventory"])
        count = len(self.state["inventory"])
        self.state["inventory"] = []
        self.state["balance"] += total
        self.state["coins_earned"] += total
        self.save()
        return True, f"Sold {count} items for {format_coins_full(total)} coins.", total

    def sorted_inventory(self, sort_by: str = "Rarity", search: str = "") -> List[dict]:
        """Return the inventory filtered by search text and sorted by the chosen key."""
        items = self.state["inventory"]
        if search:
            s = search.lower().strip()
            items = [i for i in items if s in i["name"].lower()]

        if sort_by == "Rarity":
            items = sorted(items, key=lambda i: (rarity_rank(i["rarity"]), i["value"]), reverse=True)
        elif sort_by == "Value":
            items = sorted(items, key=lambda i: i["value"], reverse=True)
        elif sort_by == "Name":
            items = sorted(items, key=lambda i: i["name"])
        elif sort_by == "Newest":
            items = sorted(items, key=lambda i: i.get("obtained_at", ""), reverse=True)
        return items

    # ------------------------------------------------------------ settings
    def update_settings(self, animation_speed: str, sound_enabled: bool, confirm_sales: bool):
        self.state["settings"]["animation_speed"] = animation_speed
        self.state["settings"]["sound_enabled"] = bool(sound_enabled)
        self.state["settings"]["confirm_sales"] = bool(confirm_sales)
        self.save()

    def reset_progress(self):
        """Wipe progress back to a fresh save (used by the Settings page)."""
        self.state.clear()
        self.state.update(default_save_data())
        self.save()

    # ------------------------------------------------------------ stats
    def rarest_item_owned(self) -> Optional[dict]:
        if not self.state["inventory"]:
            return None
        return max(self.state["inventory"], key=lambda i: (rarity_rank(i["rarity"]), i["value"]))


engine = GameEngine(STATE)
print("Game engine initialized.")
print(f"Starting balance available: {engine.balance:,} coins")

## 5. Case-Opening Scroll Animation
Builds a CS:GO-style horizontal scrolling reel as an HTML/CSS artifact that decelerates and lands on the winning item, which is then highlighted.

In [ ]:
# --------------------------------------------------------------------------
# 9. SCROLL ANIMATION (CS:GO-STYLE CASE OPENING)
# --------------------------------------------------------------------------
# We build a horizontal reel of item "cards" as HTML, then use a CSS
# transition/keyframe animation to scroll it and decelerate onto the
# winning card, which sits at a fixed target position. The winning card
# is duplicated into the reel at the correct index so the animation lands
# exactly on it.

ANIMATION_DURATION = {
    "Slow": 6.5,
    "Normal": 4.5,
    "Fast": 2.5,
}

CARD_WIDTH = 150   # px, including margin
REEL_LENGTH = 60   # number of filler cards before the winning card
WINNER_INDEX = REEL_LENGTH  # winning card sits at this position in the reel


def _item_card_html(item: dict, is_winner: bool = False) -> str:
    """Render a single reel card's HTML for a given item dict."""
    color = RARITY_COLORS.get(item["rarity"], "#ffffff")
    winner_class = "winner-card" if is_winner else ""
    glow = f"box-shadow: 0 0 25px 4px {color}99;" if is_winner else ""
    return f'''
    <div class="reel-card {winner_class}" style="border-color:{color}; {glow}">
        <div class="reel-emoji">{item["emoji"]}</div>
        <div class="reel-name" style="color:{color};">{item["name"]}</div>
        <div class="reel-value">{format_coins(item["value"])}</div>
    </div>
    '''


def build_case_opening_html(case_name: str, winning_item: dict, anim_speed: str = "Normal") -> str:
    """
    Build the full HTML/CSS/JS block for the scrolling case-opening animation.
    The reel is filled with random items drawn from the case's drop table
    (for visual variety), with the true winning item inserted at a fixed
    index so the deceleration lands exactly on it.
    """
    case = CASES[case_name]
    table_names = list(case["drop_table"].keys())

    # Build filler cards (random draws from this case's own pool for flavor)
    reel_items = []
    for _ in range(REEL_LENGTH):
        name = random.choice(table_names)
        reel_items.append(ITEM_DB[name])
    # Insert the true winner at WINNER_INDEX
    reel_items.insert(WINNER_INDEX, winning_item)
    # Pad a few extra cards after the winner so the reel doesn't visibly end
    for _ in range(8):
        name = random.choice(table_names)
        reel_items.append(ITEM_DB[name])

    cards_html = "".join(
        _item_card_html(it, is_winner=(idx == WINNER_INDEX))
        for idx, it in enumerate(reel_items)
    )

    duration = ANIMATION_DURATION.get(anim_speed, ANIMATION_DURATION["Normal"])
    # Center the winning card under the pointer: shift left by its pixel
    # offset, then re-center by half the visible reel width.
    target_offset = WINNER_INDEX * CARD_WIDTH

    html = f'''
    <div class="case-opening-wrapper">
      <div class="reel-pointer"></div>
      <div class="reel-viewport">
        <div class="reel-track" id="reel-track-{winning_item['id'] if 'id' in winning_item else 'x'}"
             style="transform: translateX(0px);
                    animation: scrollReel-{winning_item.get('id','x')} {duration}s cubic-bezier(0.12, 0.85, 0.15, 1) forwards;">
          {cards_html}
        </div>
      </div>
    </div>

    <style>
    .case-opening-wrapper {{
        position: relative;
        width: 100%;
        max-width: 900px;
        margin: 0 auto;
        overflow: hidden;
        background: linear-gradient(180deg, #1a1a2e 0%, #0f0f1a 100%);
        border-radius: 12px;
        padding: 24px 0;
        border: 1px solid #2e2e4a;
    }}
    .reel-pointer {{
        position: absolute;
        left: 50%;
        top: 0;
        bottom: 0;
        width: 3px;
        background: #ffcc00;
        box-shadow: 0 0 12px 3px #ffcc00aa;
        z-index: 10;
        transform: translateX(-50%);
    }}
    .reel-viewport {{
        overflow: hidden;
        width: 100%;
        height: 170px;
    }}
    .reel-track {{
        display: flex;
        align-items: center;
        will-change: transform;
    }}
    .reel-card {{
        flex: 0 0 auto;
        width: {CARD_WIDTH - 14}px;
        height: 150px;
        margin: 0 7px;
        background: #202038;
        border: 2px solid #444;
        border-radius: 10px;
        display: flex;
        flex-direction: column;
        align-items: center;
        justify-content: center;
        text-align: center;
        transition: box-shadow 0.3s ease;
    }}
    .reel-emoji {{ font-size: 40px; margin-bottom: 8px; }}
    .reel-name {{ font-size: 12px; font-weight: 700; padding: 0 6px; line-height: 1.2; }}
    .reel-value {{ font-size: 11px; color: #aaa; margin-top: 4px; }}
    .winner-card {{
        border-width: 3px;
        transform: scale(1.04);
    }}
    @keyframes scrollReel-{winning_item.get('id','x')} {{
        0%   {{ transform: translateX(0px); }}
        100% {{ transform: translateX(calc(-{target_offset}px + 50% - {(CARD_WIDTH-14)//2}px)); }}
    }}
    </style>
    '''
    return html


def build_result_banner_html(item: dict) -> str:
    """Build the highlighted 'you won' result banner shown after the reel stops."""
    color = RARITY_COLORS.get(item["rarity"], "#ffffff")
    return f'''
    <div style="
        margin-top: 18px;
        padding: 18px 24px;
        border-radius: 12px;
        background: linear-gradient(135deg, {color}22, #1a1a2e);
        border: 2px solid {color};
        text-align: center;
        animation: popIn 0.4s ease-out;
    ">
        <div style="font-size: 46px;">{item["emoji"]}</div>
        <div style="font-size: 22px; font-weight: 800; color: {color}; margin-top: 4px;">
            {item["name"]}
        </div>
        <div style="font-size: 15px; color: #ddd; margin-top: 2px;">
            Rarity: <span style="color:{color}; font-weight:700;">{item["rarity"]}</span>
             &nbsp;|&nbsp; Value: <span style="color:#ffd700; font-weight:700;">{format_coins_full(item["value"])} coins</span>
        </div>
    </div>
    <style>
    @keyframes popIn {{
        0%   {{ opacity: 0; transform: scale(0.85); }}
        100% {{ opacity: 1; transform: scale(1); }}
    }}
    </style>
    '''

print("Animation builders ready.")

## 6. Dark Theme CSS
A modern, smooth dark theme applied globally to the Gradio app.

In [ ]:
# --------------------------------------------------------------------------
# 10. GLOBAL DARK THEME CSS
# --------------------------------------------------------------------------
CUSTOM_CSS = """
:root {
    --bg-main: #0f0f1a;
    --bg-panel: #181826;
    --bg-card: #202038;
    --accent: #55ff9c;
    --accent2: #ffaa00;
    --text-main: #eaeaf5;
    --text-dim: #9d9dbd;
}

.gradio-container {
    background: radial-gradient(circle at 20% 0%, #1a1a30 0%, #0f0f1a 60%) !important;
    color: var(--text-main) !important;
    font-family: 'Segoe UI', 'Trebuchet MS', sans-serif !important;
}

/* Smooth transitions everywhere */
* {
    transition: background-color 0.25s ease, box-shadow 0.25s ease,
                transform 0.15s ease, border-color 0.25s ease;
}

h1, h2, h3 {
    color: var(--text-main) !important;
    text-shadow: 0 0 12px rgba(85,255,156,0.25);
}

/* Tabs */
.tabitem {
    background: var(--bg-panel) !important;
    border-radius: 12px !important;
}
button[role="tab"] {
    font-weight: 600 !important;
}
button[role="tab"][aria-selected="true"] {
    color: var(--accent) !important;
    border-bottom: 2px solid var(--accent) !important;
}

/* Buttons */
.gr-button, button {
    border-radius: 10px !important;
}
button:hover {
    transform: translateY(-1px);
    box-shadow: 0 4px 14px rgba(85,255,156,0.18);
}

/* Panels / groups */
.gr-panel, .gr-box, .gr-form {
    background: var(--bg-panel) !important;
    border: 1px solid #2b2b46 !important;
    border-radius: 14px !important;
}

/* Stat / balance display */
.balance-display {
    font-size: 26px;
    font-weight: 800;
    color: var(--accent2);
    text-shadow: 0 0 14px rgba(255,170,0,0.35);
}

.case-card {
    background: linear-gradient(160deg, #202038 0%, #16162a 100%);
    border: 1px solid #33335a;
    border-radius: 16px;
    padding: 16px;
    text-align: center;
}
.case-card:hover {
    border-color: var(--accent);
    box-shadow: 0 0 22px rgba(85,255,156,0.18);
}

.stat-box {
    background: var(--bg-card);
    border: 1px solid #2e2e4a;
    border-radius: 12px;
    padding: 14px 18px;
    text-align: center;
}
.stat-box .label { color: var(--text-dim); font-size: 13px; }
.stat-box .value { font-size: 20px; font-weight: 800; margin-top: 4px; }
"""

print("Theme CSS ready.")

## 7. UI Rendering Helpers
Functions that turn game state into HTML fragments for each page (home, shop, inventory, stats).

In [ ]:
# --------------------------------------------------------------------------
# 11. HTML RENDER HELPERS FOR EACH PAGE
# --------------------------------------------------------------------------
def render_balance_html() -> str:
    return f'''
    <div class="balance-display">💰 {format_coins_full(engine.balance)} coins</div>
    <div style="color:#9d9dbd; font-size:13px; margin-top:2px;">
        Inventory value: {format_coins_full(engine.inventory_value())} coins
    </div>
    '''


def render_home_html() -> str:
    net = engine.profit_loss()
    net_color = "#55ff9c" if net >= 0 else "#ff5555"
    best = engine.state.get("best_item")
    best_html = "None yet" if not best else (
        f'{best["emoji"]} <b style="color:{RARITY_COLORS.get(best["rarity"],"#fff")}">{best["name"]}</b> '
        f'({format_coins_full(best["value"])} coins)'
    )
    return f'''
    <div style="text-align:center; padding: 10px 0 20px 0;">
        <h1 style="font-size:34px; margin-bottom:0;">⛏️ Minecraft Case Opener</h1>
        <p style="color:#9d9dbd;">Open cases, collect legendary loot, and build your fortune in coins.</p>
    </div>
    <div style="display:flex; gap:16px; flex-wrap:wrap; justify-content:center;">
        <div class="stat-box" style="min-width:180px;">
            <div class="label">Balance</div>
            <div class="value" style="color:#ffaa00;">{format_coins(engine.balance)}</div>
        </div>
        <div class="stat-box" style="min-width:180px;">
            <div class="label">Inventory Value</div>
            <div class="value" style="color:#55ffff;">{format_coins(engine.inventory_value())}</div>
        </div>
        <div class="stat-box" style="min-width:180px;">
            <div class="label">Cases Opened</div>
            <div class="value">{engine.state['cases_opened']:,}</div>
        </div>
        <div class="stat-box" style="min-width:180px;">
            <div class="label">Profit / Loss</div>
            <div class="value" style="color:{net_color};">{format_coins(net)}</div>
        </div>
    </div>
    <div style="margin-top:22px; text-align:center;">
        <div style="color:#9d9dbd; font-size:13px;">Best item obtained</div>
        <div style="font-size:17px; margin-top:4px;">{best_html}</div>
    </div>
    '''


def render_case_shop_html() -> str:
    """Render all six cases as cards, each showing price, drop count, and rare-item preview."""
    cards = []
    for name, info in CASES.items():
        table = info["drop_table"]
        drop_count = len(table)
        # Rare preview = top 3 items by value in this case's table
        top_items = sorted(table.keys(), key=lambda n: ITEM_DB[n]["value"], reverse=True)[:3]
        preview_html = "".join(
            f'<span style="color:{RARITY_COLORS[ITEM_DB[n]["rarity"]]}; margin-right:8px;">'
            f'{ITEM_DB[n]["emoji"]} {n}</span>'
            for n in top_items
        )
        afford = "✅" if engine.can_afford(name) else "❌"
        cards.append(f'''
        <div class="case-card" style="width:270px;">
            <div style="font-size:52px;">{info["icon"]}</div>
            <div style="font-size:19px; font-weight:800; margin-top:4px;">{name}</div>
            <div style="color:#9d9dbd; font-size:12px; margin:6px 0 10px 0;">{info["description"]}</div>
            <div style="font-size:16px; color:#ffaa00; font-weight:700;">
                {format_coins_full(info["price"])} coins {afford}
            </div>
            <div style="color:#9d9dbd; font-size:12px; margin-top:4px;">{drop_count} possible drops</div>
            <div style="margin-top:10px; font-size:12px; line-height:1.8;">{preview_html}</div>
        </div>
        ''')
    return f'<div style="display:flex; gap:18px; flex-wrap:wrap; justify-content:center;">{"".join(cards)}</div>'


def render_inventory_html(sort_by: str = "Rarity", search: str = "") -> str:
    items = engine.sorted_inventory(sort_by=sort_by, search=search)
    if not items:
        return '<div style="text-align:center; color:#9d9dbd; padding:40px;">Your inventory is empty. Open some cases!</div>'

    rows = []
    for item in items:
        color = RARITY_COLORS.get(item["rarity"], "#fff")
        rows.append(f'''
        <div class="case-card" style="width:160px; border-color:{color}55;">
            <div style="font-size:34px;">{item["emoji"]}</div>
            <div style="font-weight:700; color:{color}; font-size:13px; margin-top:4px;">{item["name"]}</div>
            <div style="font-size:11px; color:#9d9dbd;">{item["rarity"]}</div>
            <div style="font-size:13px; color:#ffd700; font-weight:700; margin-top:4px;">
                {format_coins(item["value"])}
            </div>
            <div style="font-size:10px; color:#666; margin-top:4px;">ID: {item["id"][:8]}</div>
        </div>
        ''')
    return f'<div style="display:flex; gap:14px; flex-wrap:wrap; justify-content:center;">{"".join(rows)}</div>'


def render_statistics_html() -> str:
    s = engine.state
    rarest = engine.rarest_item_owned()
    rarest_html = "None owned" if not rarest else (
        f'{rarest["emoji"]} <b style="color:{RARITY_COLORS.get(rarest["rarity"],"#fff")}">{rarest["name"]}</b> '
        f'({rarest["rarity"]})'
    )
    best = s.get("best_item")
    best_html = "None yet" if not best else (
        f'{best["emoji"]} <b style="color:{RARITY_COLORS.get(best["rarity"],"#fff")}">{best["name"]}</b> '
        f'({format_coins_full(best["value"])} coins)'
    )
    net = engine.profit_loss()
    net_color = "#55ff9c" if net >= 0 else "#ff5555"

    def box(label, value, color="#eaeaf5"):
        return f'''
        <div class="stat-box" style="min-width:200px;">
            <div class="label">{label}</div>
            <div class="value" style="color:{color};">{value}</div>
        </div>
        '''

    boxes = "".join([
        box("Total Cases Opened", f'{s["cases_opened"]:,}'),
        box("Inventory Value", format_coins_full(engine.inventory_value()), "#55ffff"),
        box("Coins Spent", format_coins_full(s["coins_spent"]), "#ff8888"),
        box("Coins Earned", format_coins_full(s["coins_earned"]), "#88ff88"),
        box("Profit / Loss", format_coins_full(net), net_color),
        box("Current Balance", format_coins_full(engine.balance), "#ffaa00"),
    ])

    return f'''
    <div style="display:flex; gap:16px; flex-wrap:wrap; justify-content:center;">{boxes}</div>
    <div style="display:flex; gap:30px; justify-content:center; margin-top:24px; flex-wrap:wrap;">
        <div style="text-align:center;">
            <div style="color:#9d9dbd; font-size:13px;">🏆 Best Item Ever Obtained</div>
            <div style="font-size:17px; margin-top:6px;">{best_html}</div>
        </div>
        <div style="text-align:center;">
            <div style="color:#9d9dbd; font-size:13px;">✨ Rarest Item Currently Owned</div>
            <div style="font-size:17px; margin-top:6px;">{rarest_html}</div>
        </div>
    </div>
    '''

print("Render helpers ready.")

## 8. Gradio Application
Assembles every page into a single `Blocks` app, wires up all callbacks (buy, open, sell, sort, search, settings, save/load), and launches the server.

In [ ]:
# --------------------------------------------------------------------------
# 12. GRADIO APPLICATION
# --------------------------------------------------------------------------
CASE_NAMES = list(CASES.keys())

def refresh_topbar():
    """Refresh the persistent balance/inventory-value display."""
    return render_balance_html()


# ---- Home page callbacks ---------------------------------------------------
def refresh_home():
    return render_home_html(), refresh_topbar()


# ---- Case shop callbacks ---------------------------------------------------
def refresh_shop():
    return render_case_shop_html(), refresh_topbar()


def do_open_case(case_name: str, anim_speed: str):
    """
    Open a case: validate funds, roll the item, build the scroll animation,
    and return the animation HTML + result banner + updated topbar/shop.
    """
    if not case_name:
        empty = '<div style="text-align:center; color:#ff8888;">Select a case first.</div>'
        return empty, "", refresh_topbar(), render_case_shop_html()

    ok, msg, item = engine.open_case(case_name)
    if not ok:
        warn = f'<div style="text-align:center; color:#ff8888; padding:20px;">{msg}</div>'
        return warn, "", refresh_topbar(), render_case_shop_html()

    reel_html = build_case_opening_html(case_name, item, anim_speed)
    result_html = build_result_banner_html(item)
    return reel_html, result_html, refresh_topbar(), render_case_shop_html()


# ---- Inventory callbacks ---------------------------------------------------
def refresh_inventory(sort_by, search):
    items = engine.sorted_inventory(sort_by=sort_by, search=search)
    choices = [f'{i["name"]} | {i["rarity"]} | {format_coins(i["value"])} | {i["id"]}' for i in items]
    return render_inventory_html(sort_by, search), gr.update(choices=choices, value=None), refresh_topbar()


def do_sell_selected(selected_label, sort_by, search):
    if not selected_label:
        return render_inventory_html(sort_by, search), gr.update(), refresh_topbar(), "Select an item to sell first."
    item_id = selected_label.split("|")[-1].strip()
    ok, msg, _ = engine.sell_item(item_id)
    inv_html, dropdown_update, topbar = refresh_inventory(sort_by, search)
    return inv_html, dropdown_update, topbar, msg


def do_sell_all(sort_by, search):
    ok, msg, _ = engine.sell_all()
    inv_html, dropdown_update, topbar = refresh_inventory(sort_by, search)
    return inv_html, dropdown_update, topbar, msg


# ---- Statistics callbacks ---------------------------------------------------
def refresh_stats():
    return render_statistics_html(), refresh_topbar()


# ---- Settings callbacks ---------------------------------------------------
def save_settings(anim_speed, sound_enabled, confirm_sales):
    engine.update_settings(anim_speed, sound_enabled, confirm_sales)
    return "✅ Settings saved."


def do_manual_save():
    engine.save()
    return f"✅ Progress saved to {SAVE_PATH}"


def do_manual_reload():
    global STATE
    STATE = load_save()
    engine.state = STATE
    return "✅ Progress reloaded from disk.", refresh_topbar()


def do_reset_progress():
    engine.reset_progress()
    return ("⚠️ Progress has been reset to a fresh save.",
            refresh_topbar(), render_home_html(), render_case_shop_html(),
            render_statistics_html())


# --------------------------------------------------------------------------
# 13. BUILD THE BLOCKS APP
# --------------------------------------------------------------------------
with gr.Blocks(css=CUSTOM_CSS, theme=gr.themes.Base(primary_hue="green", neutral_hue="slate"),
                title="Minecraft Case Opener") as demo:

    gr.Markdown("# ⛏️ Minecraft Case Opener")
    topbar = gr.HTML(render_balance_html())

    with gr.Tabs():
        # ---------------------------------------------------------- HOME
        with gr.TabItem("🏠 Home"):
            home_html = gr.HTML(render_home_html())
            home_refresh_btn = gr.Button("🔄 Refresh")
            home_refresh_btn.click(fn=refresh_home, outputs=[home_html, topbar])

        # ---------------------------------------------------------- SHOP
        with gr.TabItem("🛒 Case Shop"):
            shop_html = gr.HTML(render_case_shop_html())
            gr.Markdown("### Open a Case")
            with gr.Row():
                case_dropdown = gr.Dropdown(choices=CASE_NAMES, value=CASE_NAMES[0], label="Choose a case")
                speed_dropdown = gr.Dropdown(choices=["Slow", "Normal", "Fast"], value="Normal",
                                              label="Animation speed")
                open_btn = gr.Button("🎲 Open Case", variant="primary")
            reel_html = gr.HTML("")
            result_html = gr.HTML("")

            open_btn.click(
                fn=do_open_case,
                inputs=[case_dropdown, speed_dropdown],
                outputs=[reel_html, result_html, topbar, shop_html],
            )

        # ---------------------------------------------------------- INVENTORY
        with gr.TabItem("🎒 Inventory"):
            with gr.Row():
                sort_dropdown = gr.Dropdown(choices=["Rarity", "Value", "Name", "Newest"],
                                             value="Rarity", label="Sort by")
                search_box = gr.Textbox(label="Search inventory", placeholder="Type an item name...")
                inv_refresh_btn = gr.Button("🔄 Refresh")

            inv_html = gr.HTML(render_inventory_html())
            item_selector = gr.Dropdown(choices=[], label="Select item to sell (name | rarity | value | id)")

            with gr.Row():
                sell_selected_btn = gr.Button("💸 Sell Selected", variant="secondary")
                sell_all_btn = gr.Button("🔥 Sell All", variant="stop")

            inv_message = gr.Markdown("")

            inv_refresh_btn.click(fn=refresh_inventory, inputs=[sort_dropdown, search_box],
                                   outputs=[inv_html, item_selector, topbar])
            sort_dropdown.change(fn=refresh_inventory, inputs=[sort_dropdown, search_box],
                                  outputs=[inv_html, item_selector, topbar])
            search_box.change(fn=refresh_inventory, inputs=[sort_dropdown, search_box],
                               outputs=[inv_html, item_selector, topbar])
            sell_selected_btn.click(fn=do_sell_selected, inputs=[item_selector, sort_dropdown, search_box],
                                     outputs=[inv_html, item_selector, topbar, inv_message])
            sell_all_btn.click(fn=do_sell_all, inputs=[sort_dropdown, search_box],
                                outputs=[inv_html, item_selector, topbar, inv_message])

        # ---------------------------------------------------------- STATISTICS
        with gr.TabItem("📊 Statistics"):
            stats_html = gr.HTML(render_statistics_html())
            stats_refresh_btn = gr.Button("🔄 Refresh")
            stats_refresh_btn.click(fn=refresh_stats, outputs=[stats_html, topbar])

        # ---------------------------------------------------------- SETTINGS
        with gr.TabItem("⚙️ Settings"):
            gr.Markdown("### Game Settings")
            anim_speed_setting = gr.Dropdown(choices=["Slow", "Normal", "Fast"],
                                              value=engine.state["settings"]["animation_speed"],
                                              label="Default animation speed")
            sound_setting = gr.Checkbox(value=engine.state["settings"]["sound_enabled"],
                                         label="Sound enabled (cosmetic setting)")
            confirm_setting = gr.Checkbox(value=engine.state["settings"]["confirm_sales"],
                                           label="Confirm before selling (cosmetic setting)")
            save_settings_btn = gr.Button("💾 Save Settings")
            settings_msg = gr.Markdown("")
            save_settings_btn.click(fn=save_settings,
                                     inputs=[anim_speed_setting, sound_setting, confirm_setting],
                                     outputs=[settings_msg])

            gr.Markdown("### Save Data")
            gr.Markdown(f"Save file path: `{SAVE_PATH}`")
            with gr.Row():
                manual_save_btn = gr.Button("💾 Save Progress Now")
                manual_reload_btn = gr.Button("📂 Reload From Disk")
            save_msg = gr.Markdown("")
            manual_save_btn.click(fn=do_manual_save, outputs=[save_msg])
            manual_reload_btn.click(fn=do_manual_reload, outputs=[save_msg, topbar])

            gr.Markdown("### ⚠️ Danger Zone")
            reset_btn = gr.Button("🗑️ Reset All Progress", variant="stop")
            reset_msg = gr.Markdown("")
            reset_btn.click(fn=do_reset_progress,
                             outputs=[reset_msg, topbar, home_html, shop_html, stats_html])

print("Gradio Blocks app constructed successfully.")

In [ ]:
# --------------------------------------------------------------------------
# 14. LAUNCH
# --------------------------------------------------------------------------
# share=True gives a public link (works on Kaggle); debug=False keeps logs clean.
# If a previous launch is still bound to a port, close it first to avoid conflicts.
try:
    demo.close()
except Exception:
    pass

try:
    demo.launch(share=True, debug=False, show_error=True)
except Exception as e:
    print(f"Launch with share=True failed ({e}); retrying without share link...")
    demo.launch(share=False, debug=False, show_error=True)